# Heritability: aGTEx / dGTEx data prep

Pipeline overview:

1. Load donor metadata and sample attributes
2. Locate the expression (TPM) inputs and define the tissue sets
3. Build per-tissue PLINK `.fam` files with gene expression as phenotypes
4. Sample sizes per tissue (RNA-seq ∩ WGS)
5. Choose a young / old age threshold
6. Predicted-expression models (peek)
7. dGTEx

## 0. Setup

In [ ]:
import os
from itertools import compress  # (kept from original imports; not currently used)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# ---- Paths -----------------------------------------------------------------
BASE              = r'R:\heritability'
TPM_DIR           = os.path.join(BASE, 'aGTEx_gene_TPMs_by_tissue')
PLINK_DIR         = os.path.join(BASE, 'plink_qc_vcf')
FAM_EXP_DIR       = os.path.join(BASE, 'fam_files_exp')        # output of section 3
PREDICTED_EXP_DIR = os.path.join(BASE, 'predicted_exp')

# Relative to the notebook's working directory
UPDATED_FAM_DIR   = 'updated_fam_files'   # per-tissue .fam files (all aGTEx tissues), used in section 4
AGE_HIST_DIR      = 'age_hists'
DGTEX_METADATA    = 'dgtex_metadata.xlsx'
DGTEX_TPM         = 'dGTEx_gene_tpm.gct.gz'

# ---- TPM file naming -------------------------------------------------------
TPM_PREFIX = 'gene_tpm_2017-06-05_v8_'
TPM_SUFFIX = '.gct.gz'

# Tissues that have phenotype files / are used for the GEMMA runs
VALID_TISSUES = [
    'adipose_subcutaneous', 'artery_tibial', 'esophagus_mucosa', 'esophagus_muscularis',
    'muscle_skeletal', 'nerve_tibial', 'skin_sun_exposed_lower_leg', 'whole_blood',
]

In [ ]:
def tissue_from_file(tpm_file):
    """'gene_tpm_2017-06-05_v8_artery_tibial.gct.gz' -> 'artery_tibial'"""
    return tpm_file.split(TPM_PREFIX)[1].split(TPM_SUFFIX)[0]


def short_id(sample_id):
    """Full GTEx sample ID -> donor ID: 'GTEX-1117F-0226-SM-5GZZ7' -> 'GTEX-1117F'"""
    return '-'.join(sample_id.split('-')[:2])

## 1. Donor metadata and sample attributes

In [ ]:
agtex_metadata = pd.read_table(os.path.join(BASE, 'agtex_metadata.txt'))
agtex_attrs    = pd.read_table(os.path.join(BASE, 'agtex_attrs.txt'), low_memory=False)

# The first column ('Unnamed: 0') is the full sample ID; derive the donor ID from it
agtex_attrs.insert(0, 'donor_id', 'GTEX-' + agtex_attrs['Unnamed: 0'].str.split('-', expand=True)[1])

print('unique donors in sample attributes:', agtex_attrs['donor_id'].nunique())

## 2. Expression inputs and tissue sets

In [ ]:
gene_list = pd.read_table(os.path.join(BASE, 'gene_column_mapping.txt'), header=None)[0].tolist()

tpm_files       = os.listdir(TPM_DIR)
agtex_tissues   = [tissue_from_file(f) for f in tpm_files]
valid_tpm_files = [f for f in tpm_files if tissue_from_file(f) in VALID_TISSUES]

print(f'{len(gene_list)} genes | {len(tpm_files)} tissues | {len(valid_tpm_files)} tissues with pheno files')

## 3. Per-tissue `.fam` files with expression phenotypes

The base `.fam` (838 donors with WGS/genotype data) is merged with each tissue's TPM values, one extra column per gene.
Only donors with both genotypes and RNA-seq in that tissue are kept.

### 3.1 Base `.fam`

In [ ]:
base_fam = pd.read_table(os.path.join(PLINK_DIR, 'agtex_chr1_qc_noambig.fam'), sep=' ', header=None)
# columns: 0=FID, 1=IID, 2=father, 3=mother, 4=sex, 5=phenotype (-9 placeholder)

print('donors with genotypes:', len(base_fam))
print('  ...of which in metadata:', base_fam[1].isin(agtex_metadata['SUBJID']).sum())

### 3.2 QC: donor overlap

The aGTEx TPM files contain donor IDs that are not among the 838 donors with VCF / genotype data, so the merge below is an inner join.
Also checks that each tissue has at most one sample per donor (needed for the donor-level join).
Only the header row of each TPM file is read, so this is fast.

In [ ]:
def tpm_sample_ids(tpm_file):
    """Sample IDs (columns) of a TPM file, without loading the expression matrix."""
    header = pd.read_csv(os.path.join(TPM_DIR, tpm_file), sep='\t', skiprows=2, nrows=0,
                         compression='gzip', index_col=0)
    return list(header.columns[2:])  # columns are: Description, then one column per sample (Name is dropped by index_col=0)


qc_rows = []
for tpm_file in valid_tpm_files:
    donors = pd.Series([short_id(s) for s in tpm_sample_ids(tpm_file)])
    qc_rows.append({
        'tissue': tissue_from_file(tpm_file),
        'n_rnaseq_samples': len(donors),
        'one_sample_per_donor': donors.is_unique,
        'n_with_genotypes': int(donors.isin(base_fam[1]).sum()),
    })
pd.DataFrame(qc_rows)

### 3.3 Build and write the `.fam` files

In [ ]:
def build_expression_fam(tpm_file, genes, base_fam):
    """Merge a tissue's TPM values (columns = `genes`) onto `base_fam`.

    Returns the merged table and the gene order of the expression columns.
    """
    tpm = pd.read_csv(os.path.join(TPM_DIR, tpm_file), sep='\t', skiprows=2,
                      compression='gzip', index_col=0)
    tpm = (tpm[tpm['Description'].isin(genes)]
           .drop_duplicates(subset='Description')
           .set_index('Description'))

    expr = tpm.drop(columns='Name').T          # rows = samples, columns = genes
    expr.index = [short_id(s) for s in expr.index]

    fam = base_fam.merge(expr, left_on=0, right_index=True)   # inner join on donor ID
    return fam, list(expr.columns)

In [ ]:
os.makedirs(FAM_EXP_DIR, exist_ok=True)

reference_genes = None
for tpm_file in valid_tpm_files:
    tissue = tissue_from_file(tpm_file)
    fam, genes = build_expression_fam(tpm_file, gene_list, base_fam)

    # every tissue must have the same gene columns in the same order
    if reference_genes is None:
        reference_genes = genes
    assert genes == reference_genes, f'gene columns differ for {tissue}'

    fam.to_csv(os.path.join(FAM_EXP_DIR, f'{tissue}_agtex.fam'), sep=' ', header=False, index=False)
    print(f'{tissue}: {fam.shape[0]} donors x {len(genes)} genes')

### 3.4 Gene → column mapping

Column index of each gene's phenotype (first gene = 2, since the `.fam`'s own placeholder phenotype is column 1).
Written to the working directory.

In [ ]:
pd.DataFrame({'gene': reference_genes,
              'column': np.arange(2, len(reference_genes) + 2)}
             ).to_csv('gene_column_mapping.txt', sep='\t', header=False, index=False)

## 4. Sample sizes per tissue (RNA-seq ∩ WGS)

In [ ]:
# All RNA-seq samples per tissue (no WGS filter)
rna_df = agtex_attrs['SMTSD'].value_counts().rename_axis('tissue').reset_index(name='n')

In [ ]:
rows = []
for tissue in agtex_tissues:
    fam = pd.read_table(os.path.join(UPDATED_FAM_DIR, f'{tissue}_agtex.fam'), sep=' ', header=None)
    md_t = agtex_metadata[agtex_metadata['SUBJID'].isin(fam[1])]
    rows.append({
        'tissue': tissue,
        'n': len(md_t),
        'n_young': int((md_t['AGE'] <= 30).sum()),
        'n_old':   int((md_t['AGE'] >= 40).sum()),
    })
rna_and_wgs_df = pd.DataFrame(rows)

print('total tissue-donor pairs:', rna_and_wgs_df['n'].sum())
rna_and_wgs_df

## 5. Young / old age threshold

Goal: find a donor-age cutoff that leaves at least 100 "young" and 100 "old" donors (RNA-seq + WGS) in as many tissues as possible.
Tissue names here are the sample-attribute names (`SMTSD`), not the TPM file names.

In [ ]:
donors_w_wgs  = set(base_fam[1])
smtsd_tissues = sorted(agtex_attrs['SMTSD'].dropna().unique())


def wgs_donor_ages(tissue):
    """Ages of donors with both RNA-seq in `tissue` and WGS."""
    donors = agtex_attrs.loc[agtex_attrs['SMTSD'] == tissue, 'donor_id']
    donors = donors[donors.isin(donors_w_wgs)]
    return agtex_metadata.loc[agtex_metadata['SUBJID'].isin(donors), 'AGE']


ages_by_tissue = {t: wgs_donor_ages(t) for t in smtsd_tissues}

### 5.1 Cumulative age distributions (one plot per tissue)

In [ ]:
os.makedirs(AGE_HIST_DIR, exist_ok=True)

for tissue, ages in ages_by_tissue.items():
    fig, ax = plt.subplots()
    ax.hist(ages, cumulative=True, edgecolor='black', bins=range(20, 75, 5))
    ax.axhline(y=100, color='red', linestyle='--', linewidth=2)
    ax.set_title(tissue)
    fig.savefig(os.path.join(AGE_HIST_DIR, f'age_hist_{tissue}.png'))
    plt.close(fig)

### 5.2 Age at which 100 donors are younger, per tissue

`threshold_100` is the age of the 101st-youngest donor; tissues with ≥ 200 donors only.

In [ ]:
rows = []
for tissue, ages in ages_by_tissue.items():
    sorted_ages = ages.sort_values().reset_index(drop=True)
    rows.append({
        'tissue': tissue,
        'n': len(sorted_ages),
        'threshold_100': sorted_ages[100] if len(sorted_ages) > 100 else np.nan,
    })

threshold_df = pd.DataFrame(rows).dropna()
threshold_df = threshold_df[threshold_df['n'] >= 200].reset_index(drop=True)
threshold_df['threshold_100'].describe()

### 5.3 Number of tissues passing (≥ 100 young and ≥ 100 old) at each threshold

In [ ]:
def n_tissues_passing(threshold, min_per_group=100):
    n_pass = 0
    for ages in ages_by_tissue.values():
        if (ages < threshold).sum() >= min_per_group and (ages >= threshold).sum() >= min_per_group:
            n_pass += 1
    return n_pass


thresholds    = np.arange(20, 70)
number_passed = [n_tissues_passing(th) for th in thresholds]

plt.plot(thresholds, number_passed)
plt.xlabel('Threshold used to assign donors "young" and "old"')
plt.ylabel('# of tissues with at least 100 young and old donors')
plt.show()

print('max tissues passing:', max(number_passed),
      '| at thresholds:', list(thresholds[np.array(number_passed) == max(number_passed)]))

## 6. Predicted-expression models (peek)

In [ ]:
os.listdir(PREDICTED_EXP_DIR)

In [ ]:
pd.read_table(os.path.join(PREDICTED_EXP_DIR, 'reference.agtex.Adipose_Subcutaneous.all.models.txt'))

## 7. dGTEx

In [ ]:
dgtex_metadata = pd.read_excel(DGTEX_METADATA)

dgtex_tissues_subset = [
    'Colon - Sigmoid', 'Ovary', 'Pancreas', 'Pituitary', 'Spleen', 'Stomach', 'Testis',
    'Thyroid', 'Uterus', 'Adrenal Gland', 'Small Intestine - Terminal Ileum',
]
dgtex_metadata[dgtex_metadata['Tissue Site Detail'].isin(dgtex_tissues_subset)]['Tissue Site Detail'].value_counts()

In [ ]:
dgtex_tpm = pd.read_csv(DGTEX_TPM, sep='\t', skiprows=2, compression='gzip', index_col=0)
dgtex_tpm.head()